In [1]:
import folium
import pandas as pd

In [2]:
df=pd.read_csv('spacex_launch_geo.csv')
df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [3]:
launch_map = folium.Map()
launch_sites = folium.map.FeatureGroup()

coordinates = df[['Launch Site', 'Lat', 'Long']].drop_duplicates()
coordinates

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
26,VAFB SLC-4E,34.632834,-120.610745
36,KSC LC-39A,28.573255,-80.646895
49,CCAFS SLC-40,28.563197,-80.576820


In [4]:
for lat, lng, in zip(coordinates['Lat'], coordinates['Long']):
    launch_sites.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5,
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )

latitudes = list(coordinates.Lat)
longitudes = list(coordinates.Long)
label = list(coordinates['Launch Site'])

latitudes, longitudes, label
#launch_map

([28.56230197, 34.63283416, 28.57325457, 28.56319718],
 [-80.57735648, -120.6107455, -80.64689529, -80.57682003],
 ['CCAFS LC-40', 'VAFB SLC-4E', 'KSC LC-39A', 'CCAFS SLC-40'])

In [5]:


for late, lon, lab in zip(latitudes, longitudes, label):
    folium.Marker([late, lon], popup=lab).add_to(launch_map)

launch_map.add_child(launch_sites)

launch_map


In [6]:
spacex_df = df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [7]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [8]:
from folium.features import DivIcon

circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [9]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
for latitude, longitude, label in zip(launch_sites_df['Lat'], launch_sites_df['Long'], launch_sites_df['Launch Site']):
    coord = [latitude, longitude]
    launch_circle = folium.Circle(coord, radius=1000, color='yellow', fill=True, fill_color='blue').add_child(
        folium.Popup(label))
    marker = folium.Marker(
        coord,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % label,
        )
    )

    site_map.add_child(launch_circle)
    site_map.add_child(marker)

In [10]:
site_map

Visualize the success and failed launches for each site on the map

In [11]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [12]:
from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else "red")
spacex_df[['class','marker_color']].head()

/var/folders/02/033ry4kj3sdg5g4s60wqsnkr0000gr/T/ipykernel_6413/1137738701.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else "red")


,class,marker_color
0,0,red
1,0,red
2,0,red
3,0,red
4,0,red


In [13]:
spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [14]:
for lat, lng, mark in zip(spacex_df['Lat'], spacex_df['Long'], spacex_df['marker_color']):
    coordinate = [lat, lng]
    marker = folium.Marker(
        coordinate,
        icon=folium.Icon(
            color='white',
            icon_color=mark
        )

    )
    marker_cluster.add_child(marker)

site_map.add_child(marker_cluster)

Calculate the distance between a launch site and its proximities

In [15]:
from folium.plugins import MousePosition

formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [16]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [17]:
distance_coastline = calculate_distance(28.57303, -80.64752, 28.55505, -80.61253 )

In [20]:
coordi = [28.55505, -80.61253]

distance_marker = folium.Marker(
    coordi,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % f"{distance_coastline:10.2f} KM"
    )
)

site_map.add_child(distance_marker)

In [23]:
lines = folium.PolyLine(locations=[[28.57303, -80.64752], coordi], weight=1)

site_map.add_child(lines)
site_map

In [25]:
launch_coord = [28.57303, -80.64752]
highway_coord = [28.56345, -80.65544]
city_coord = [28.61218, -80.80959]
rail_coord = [28.57337, -80.80444]

distance_highway = calculate_distance(launch_coord[0], launch_coord[1], highway_coord[0], highway_coord[1])
distance_rail = calculate_distance(launch_coord[0], launch_coord[1], rail_coord[0], rail_coord[1])
distance_city = calculate_distance(launch_coord[0], launch_coord[1], city_coord[0], city_coord[1])

for loc, dist in zip([highway_coord, city_coord, rail_coord],[distance_highway, distance_city, distance_rail]):
    distance_marker = folium.Marker(
        loc,
        icon=DivIcon(
                icon_size=(20, 20),
                icon_anchor=(0,0),
                html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % f"{dist:10.2f} KM"
            )
        
    )

    lines = folium.PolyLine(locations=[launch_coord, loc], weight=1)

    site_map.add_child(distance_marker)
    site_map.add_child(lines)



In [26]:
site_map